# 重构：在项目中使用 SQLite



## 这一节的目标

上一节我们已经学习了 SQLite 数据库的用法，但是 zero-to-tech 项目现在仍然使用 `history.json` 存储数据。

这一节只改造存储层：会修改一些 Python 代码，但不会破坏 `/api/analyze` 和 `/api/history` 的接口约定，前端不需要修改。

改造分两步：

1. 把存储层从 `main.py` 搬到 `storage.py`，功能保持不变，仍然使用 `history.json`。
2. 把 `storage.py` 内部的文件实现换成 SQLite 实现，`main.py` 几乎不用改。


## 先给 `main.py` 分层

打开 `backend/main.py`，会发现里面有三种职责：

~~~python
import json                         # 存储
from datetime import datetime, timezone  # 业务

app = FastAPI()                     # 启动 / 配置

HISTORY_FILE = "history.json"       # 存储
def load_history(): ...             # 存储
def save_record(record): ...         # 存储

@app.post("/api/analyze")           # 接口
def analyze(req: AnalyzeRequest):
    score = ...                     # 业务
    save_record(result)             # 存储
    return result

@app.get("/api/history")            # 接口
def history():
    records = load_history()        # 存储
    records.reverse()               # 存储
    return records[:10]              # 存储
~~~

三层各管各的：

- 接口层：对接前端 API，管“接收什么、响应什么”。
- 业务层：算情感分、算拼音、拼出一条记录，管“这件事怎么做”。
- 存储层：管数据存在哪里、怎么存、怎么取。


## 为什么要先重构？

`/api/history` 里的“读出全部、倒过来、切前十”是存储层的工作，却直接写在接口函数里。

如果现在直接把文件换成数据库，就必须修改接口函数。先把存储职责框出来，之后替换实现时，`main.py` 就不用跟着变化。

目前项目很小，接口层和业务层都只有几行，不需要为了分层而拆出更多文件。最值得拆的是即将被整体替换的存储层。

## 重构方案

重构的铁律是：

> 功能一点都不能变。重构只挪位置、不改行为；外部可观察行为改变，就不叫重构。

合适的目录结构是：

~~~text
backend/
├── main.py       # 接口层 + 业务层
└── storage.py    # 存储层
~~~

文件名使用 `storage.py`，因为它表示职责，而不是某一种实现。以后即使从文件换成 SQLite，存储层的名字仍然成立。

本节严格分成“搬家”和“装修”两步：先验证搬家没有改变功能，再单独验证 SQLite 替换。

## 第一步：搬家（功能零变化）

在 `backend/` 中新建 `storage.py`，把 `main.py` 里属于存储层的代码原样剪切过来：

~~~python
# backend/storage.py
import json

HISTORY_FILE = "history.json"

def load_history():
    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []

def save_record(record):
    records = load_history()
    records.append(record)

    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def get_history():
    records = load_history()
    records.reverse()
    return records[:10]
~~~

`load_history` 和 `save_record` 一个字都没有改。`get_history` 是新函数，把原来接口函数里的三行原样包起来。数字 `10` 也照抄，搬家就是搬家，一个字都不改。

## 搬家后修改 `main.py`

只做四件事：

1. 把 `import json` 换成 `from storage import save_record, get_history`。
2. 删除 `HISTORY_FILE`、`load_history`、`save_record`。
3. 把 `/api/history` 瘦身成 `return get_history()`。
4. 其他内容一个字都不动。

`from datetime import datetime, timezone` 要保留，因为时间戳由 `analyze` 生成，属于业务层。

## 搬家后的 `main.py`

完整形状如下。除了存储层 import、存储定义和历史接口，其他内容保持原项目代码：

~~~python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pypinyin import lazy_pinyin, Style
from snownlp import SnowNLP
from datetime import datetime, timezone
from storage import save_record, get_history

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["GET", "POST"],
)

class AnalyzeRequest(BaseModel):
    text: str

def score_label(score):
    if score >= 0.6:
        return "偏积极"
    elif score <= 0.4:
        return "偏消极"
    return "中性"

@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    text = req.text
    score = round(SnowNLP(text).sentiments, 2)
    result = {
        "text": text,
        "score": score,
        "label": score_label(score),
        "pinyin": " ".join(lazy_pinyin(text, style=Style.TONE)),
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    save_record(result)
    return result

@app.get("/api/history")
def history():
    return get_history()
~~~


## `import` 是怎样工作的？

启动应用时，uvicorn 仍然寻找 `main.py`。但这一行会让同目录下的 `storage.py` 被加载：

~~~python
from storage import save_record, get_history
~~~

`storage.py` 必须建在 `backend/` 目录下，和 `main.py` 放在一起。搬家后启动后端，访问文字实验室和 `/api/history`；如果效果和之前完全一样，就说明重构成功。

## 把数量决定权交给调用方

搬家后的 `get_history` 还有一个小问题：

~~~python
def get_history():
    records = load_history()
    records.reverse()
    return records[:10]
~~~

“切一部分”是存储层的动作，但“切在 10 这个位置”是调用方的决定。存储层不应该知道用户是在手机还是电脑上查看，也不应该写死展示数量。

改成：

~~~python
# storage.py
def get_history(limit):
    records = load_history()
    records.reverse()
    return records[:limit]
~~~

~~~python
# main.py
@app.get("/api/history")
def history():
    return get_history(10)
~~~

功能仍然不变，变化的是决定的归属。这同样是一种重构。

## 第二步：装修，把文件存储换成 SQLite

现在 `storage.py` 的边界清楚了，但内部还是文件实现。接下来只改 `storage.py`，一共六处：

| # | 文件版 | SQLite 版 |
| --- | --- | --- |
| 1 | `import json` | `import sqlite3` |
| 2 | `HISTORY_FILE` | `DB_FILE` 和 `get_conn()` |
| 3 | 没有对应代码 | `init_db()`，启动时建表 |
| 4 | `save_record`：读全部、追加、整个写回 | 一句 `INSERT` |
| 5 | `get_history`：全量读、倒序、切片 | 一句 `SELECT ... ORDER BY ... LIMIT` |
| 6 | `load_history` | 删除 |


### 第 1 处：换 import

把：

~~~python
import json
~~~

换成：

~~~python
import sqlite3
~~~

换成数据库后，不再需要 `json` 和 `open()`。

### 第 2 处：从文件名到连接

文件版只有一个文件名：

~~~python
HISTORY_FILE = "history.json"
~~~

SQLite 需要先连接数据库：

~~~python
DB_FILE = "history.db"

def get_conn():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn
~~~

`row_factory` 让查询结果带上列名。默认结果类似元组；设置后可以用 `dict(row)` 转成带字段名的字典，正好作为前端 JSON。

`get_conn()` 规定如何连接数据库。每个函数用时打开连接，用完关闭。

### 第 3 处：建表

关系型数据库要先定义表结构。新增：

~~~python
def init_db():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT,
            score REAL,
            label TEXT,
            pinyin TEXT,
            created_at TEXT
        )
    """)
    conn.commit()
    conn.close()
~~~

`history` 表的字段对应上一节文件记录。`id` 是自增主键。`CREATE TABLE IF NOT EXISTS` 表示没有表时创建，已存在时跳过。

### 第 4 处：`save_record` 换成 INSERT

文件版是“读出整个文件、追加一条、整个写回”：

~~~python
def save_record(record):
    records = load_history()
    records.append(record)
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
~~~

SQLite 版只插入一行：

~~~python
def save_record(record):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO history (text, score, label, pinyin, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        [
            record["text"],
            record["score"],
            record["label"],
            record["pinyin"],
            record["created_at"],
        ],
    )
    conn.commit()
    conn.close()
~~~

值全部使用 `?` 占位符，通过参数列表交给 `execute`，不把用户输入拼进 SQL。时间戳继续沿用业务层生成的那一份。

### 第 5 处：`get_history`——三行变一句

文件版：

~~~python
def get_history(limit):
    records = load_history()
    records.reverse()
    return records[:limit]
~~~

SQLite 版：

~~~python
def get_history(limit):
    conn = get_conn()
    cur = conn.cursor()
    rows = cur.execute(
        """
        SELECT * FROM history
        ORDER BY created_at DESC
        LIMIT ?
        """,
        [limit],
    ).fetchall()
    conn.close()

    records = []
    for row in rows:
        records.append(dict(row))
    return records
~~~

`LIMIT` 后面的数字也走 `?` 占位符，值永远走参数。`fetchall()` 取回 SQL 查询结果的全部行；如果 `LIMIT` 是 10，就只取回 10 条。

### 第 6 处：删除 `load_history`

数据库版不再需要把整份数据读进内存，因此删掉：

~~~python
def load_history():
    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []
~~~

整个 `storage.py` 中不再出现 `json`、`open()` 或 `history.json`。

## 换完后的 `storage.py`（暂时版本）

六处改完后，`storage.py` 应该是：

~~~python
# backend/storage.py
import sqlite3

DB_FILE = "history.db"

def get_conn():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT,
            score REAL,
            label TEXT,
            pinyin TEXT,
            created_at TEXT
        )
    """)
    conn.commit()
    conn.close()

def save_record(record):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO history (text, score, label, pinyin, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        [
            record["text"],
            record["score"],
            record["label"],
            record["pinyin"],
            record["created_at"],
        ],
    )
    conn.commit()
    conn.close()

def get_history(limit):
    conn = get_conn()
    cur = conn.cursor()
    rows = cur.execute(
        """
        SELECT * FROM history
        ORDER BY created_at DESC
        LIMIT ?
        """,
        [limit],
    ).fetchall()
    conn.close()

    return [dict(row) for row in rows]
~~~


## `main.py` 只多两行

`storage.py` 换成 SQLite 后，`main.py` 只需要：

~~~python
from storage import init_db, save_record, get_history

app = FastAPI()

# ... 原有 CORS 和其他代码 ...

init_db()
~~~

第一行把 `init_db` 加入 import，第二行在启动时确保数据库和表存在。第一次启动会创建 `history.db` 和 `history` 表，之后会跳过已存在的表。

## 验证切换成功

1. 打开文字实验室，分析一句话。`/api/analyze` 的路径、请求体、返回字段都没有改，结果应照常返回。
2. 请求 `/api/history`，确认数据库中出现刚才的分析记录。
3. 用 DB Browser for SQLite 打开 `backend/history.db`，在 Browse Data 中查看 `history` 表。

`/api/history` 的返回会多出一个 `id` 字段，因为 `SELECT *` 把主键也查出来了。增加字段不破坏原有接口约定，下一节还会用到这个 `id`。

## “换芯不换壳”

这一节没有修改前端文件。存储从 JSON 文件换成 SQLite，页面上却没有变化：

- 两个 HTTP 接口 `/api/analyze` 和 `/api/history` 没变。
- `save_record()` 和 `get_history()` 的名字和参数没变，`analyze` 不需要修改。
- `storage.py` 对外的样子没变，`main.py` 只多了初始化数据库的代码。

只要边界立得住，边界后面的实现就可以整个替换。

## 善后第一件事：数据库文件不进 Git

在 `.gitignore` 中加入：

~~~gitignore
backend/*.db
~~~

数据库是运行时数据，每台机器和每个环境都应该有自己的数据库文件。它和代码不是一类东西。

之前的同类规则是：`node_modules`、`.venv` 等依赖不进 Git，`out`、`.next` 等产物不进 Git。

## 善后第二件事：`history.json` 退役

文件存储已经被 SQLite 替代，`history.json` 可以删除。

真实项目中的数据迁移需要保证老数据不丢、格式正确。这个项目还没有上线，历史数据不重要，从零开始最干净、最省事。

## 善后第三件事：认识 ORM

真实项目里，很多人使用 ORM（对象关系映射），例如 Python 的 SQLAlchemy。

ORM 把数据库表包装成 Python 对象，由框架帮助生成 SQL。本节只有两句 SQL，手写反而更清楚，所以不展开。知道这个概念即可。

## 善后第四件事：给 `created_at` 建索引

历史接口每次都按 `created_at` 倒序查询：

~~~sql
SELECT * FROM history
ORDER BY created_at DESC
LIMIT ?
~~~

`created_at` 是经常用于排序的列，适合建立索引。在 `init_db()` 的建表语句后加入：

~~~python
cur.execute(
    "CREATE INDEX IF NOT EXISTS idx_history_created "
    "ON history(created_at)"
)
~~~

`IF NOT EXISTS` 表示第一次启动时创建，之后跳过。`idx_history_created` 是 `idx_表名_列名` 的常见命名方式。

索引会占用额外空间，插入、更新、删除时也需要维护；不要给每一列都建立索引。

## 最终版 `init_db()`

加入索引后：

~~~python
def init_db():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT,
            score REAL,
            label TEXT,
            pinyin TEXT,
            created_at TEXT
        )
    """)
    cur.execute(
        "CREATE INDEX IF NOT EXISTS idx_history_created "
        "ON history(created_at)"
    )
    conn.commit()
    conn.close()
~~~

重启后端，索引才会真正建立。用 DB Browser 打开 `history.db`，在 Database Structure 中展开 `history` 表的 `Indices`，可以看到 `idx_history_created`。

数据量很小时，全表扫描已经很快，感受不到索引提升；索引真正省时间，通常要到几十万、几百万条数据。

## 验证清单

~~~text
1. 后端能够正常启动，并生成 history.db。
2. 文字实验室仍然能够调用 /api/analyze。
3. 分析结果仍然包含 text、score、label、pinyin、created_at。
4. /api/history 返回最新记录，数量由 main.py 传入的 10 控制。
5. DB Browser 能看到 history 表、记录和 idx_history_created 索引。
6. 项目中不再依赖 history.json。
~~~


## 还差一步：让历史“分到每个人”

存储层升级完成，历史记录已经落进 `history.db`。但现在仍然是全站一份，所有访客的记录混在同一张表里，`/api/history` 查的是所有用户的记录。

下一节会讲如何通过会话，让每个访客只看到自己的历史记录。

← 上一节：模块 6.4 数据库正传 | 下一节：模块 6.6 状态与会话


## 总结

这节课只围绕一件事：把文件版存储换成 SQLite 版，同时保持接口不变。

~~~text
先分层
  → 把存储代码搬到 storage.py
  → 先验证功能零变化
  → 再在 storage.py 内部换成 SQLite
  → main.py 引入 init_db 并在启动时调用
  → 用 DB Browser 验证表、数据和索引
~~~

关键规则：

- 重构先搬家，功能不能变；换实现后再单独验证。
- 存储层按职责命名，不按具体实现命名。
- 用户看到的是接口，接口后面的存储实现可以替换。
- 数据库文件是运行时数据，应加入 `.gitignore`。
- 经常用于排序的 `created_at` 列可以建立索引。

下一节处理“每个人只看到自己的历史记录”。